<img src="https://github.com/comet-ml/opik/blob/main/apps/opik-documentation/documentation/static/img/opik-logo.svg?raw=true" width="200" height="100" alt="Opik Logo">

# **Opik Production Monitoring Workshop**

## **Overview**
This session provides will provide you with an understanding of how you can use Opik to monitor your applications in production through **online evaluations, dashboards, and alerting**. In this session you will:
- Configure Opik with your API Key and Workspace
- Build a CRM agent that uses OpenAI
- Trace all OpenAI calls, tool calls, and functions in Opik
- Ingest production traces from the past month
- Add online evaluations to threads and traces
- View the performance over time through dashboards
- Understand how to set up webhook alerts

## Setup & Configuration

In [ ]:
%pip install opik langchain-openai langgraph --q

Configure Opik with your API Key and personal workspace

In [ ]:
import opik
opik.configure(use_local=False, force=True)

In [ ]:
import os
import json
import getpass
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from opik.integrations.langchain import OpikTracer
import opik

PROJECT_NAME = "CRM-Chatbot-Agent-Opik-Prod"
os.environ["PROJECT_NAME"] = PROJECT_NAME

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

llm = ChatOpenAI(model="gpt-4o")


class CRMState(TypedDict):
    query: str
    system_prompt: str
    category: str
    context: str
    response: str

## Tracing our Agent



### Defining the Agent

This workshop uses a custom **CRM Agent** built with **LangGraph** and **LangChain**. The agent is designed to handle customer relationship queries by routing them to one of three specialized tool nodes:

1. **Contact Insight Tool** - Retrieves information about contacts, accounts, and deals
2. **Sales Analytics Tool** - Provides forecasts, KPIs, and performance metrics
3. **Fallback Tool** - Handles queries outside the CRM domain

The agent follows this architecture:

<img src="https://i.imgur.com/O8kDKmQ.png" width="400">

Each step of the agent is automatically traced via Opik's `OpikTracer` callback, giving you full visibility into the agent's decision-making process.

## Building the Agent with LangGraph

We will use **LangGraph** to define the agent as a state graph, with Opik's `OpikTracer` callback for automatic tracing. The agent consists of:
- A `CRMState` definition to track data through the pipeline
- A query classifier node to route requests
- Three specialized tool nodes (contact, sales, fallback)
- A RAG retrieval step
- A response formatter node

**Define RAG Context**

Static contact and sales context that will be retrieved based on the query classification.

In [ ]:
import json

CONTACT_CONTEXT = {
  "customer": {
    "contact": {
      "name": "Jessica Lau",
      "title": "Head of IT Procurement",
      "email": "jessica.lau@greenleaftech.com",
      "status": "High-value prospect"
    },
    "company": "Greenleaf Tech",
    "industry": "Renewable Energy Tech",
    "deal_stage": "Proposal Sent",
    "deal_size_estimate": 85000,
    "lead_source": "Webinar sign-up",
    "lead_source_date": "2025-03-15",
    "last_contacted": "2025-05-12",
    "engagement": {
      "emails_opened": 6,
      "emails_sent": 7,
      "average_email_open_time_hours": 2,
      "outreach_responses": 3,
      "outreach_sent": 5,
      "demos_attended": ["2025-04-04", "2025-05-09"],
      "downloads": [{"title": "Energy Grid Optimization", "date": "2025-04-17"}]
    },
    "internal_notes": [
      "Interested in predictive maintenance features",
      "Has decision authority but awaiting CFO sign-off",
      "Expressed need to implement Q3 (starting July)"
    ]
  }
}

SALES_CONTEXT = {
  "customer": {
    "company": "NovaHealth Systems",
    "industry": "Healthcare Technology",
    "region": "Northeast US",
    "annual_revenue": 12000000,
    "account_manager": "Derek Chen",
    "deal_history": [
      {"deal_id": "DH-1007", "closed_date": "2024-11-20", "value": 45000, "product": "EHR Integration Suite", "stage": "Closed Won", "sales_cycle_days": 38},
      {"deal_id": "DH-1018", "closed_date": "2025-05-06", "value": 67000, "product": "AI Diagnostics Module", "stage": "Closed Won", "sales_cycle_days": 54}
    ],
    "current_pipeline": [
      {"deal_id": "DH-1029", "product": "Mobile Patient Portal", "stage": "Negotiation", "expected_close_date": "2025-06-28", "expected_value": 85000, "sales_cycle_days_so_far": 29}
    ],
    "metrics": {
      "total_closed_value_last_12_months": 112000,
      "average_sales_cycle_days": 46,
      "close_rate": 0.66,
      "average_deal_size": 56000
    },
    "last_activity": "2025-06-06"
  }
}

**Retrieve Documents (RAG)**

Returns the appropriate context based on the query category.

In [ ]:
def rag_lookup(category):
    if category == "contact":
        return json.dumps(CONTACT_CONTEXT, indent=4)
    elif category == "sales":
        return json.dumps(SALES_CONTEXT, indent=4)
    else:
        return "No context available."

**Define Tool Nodes**

Three specialized LangGraph nodes handle different query types:
- `contact_insight_tool` — information about people, accounts, and deals
- `sales_analytics_tool` — forecasts, KPIs, and performance metrics
- `fallback_tool` — handles out-of-scope queries

In [ ]:
def contact_insight_tool(state: CRMState) -> dict:
    context = rag_lookup("contact")
    return {"context": context}


def sales_analytics_tool(state: CRMState) -> dict:
    context = rag_lookup("sales")
    return {"context": context}


def fallback_tool(state: CRMState) -> dict:
    return {"context": "No relevant context available for this query."}

**Define Routing Logic**

Routes the classified query to the appropriate tool node. LangGraph will automatically generate the graph visualization in Opik.

In [ ]:
def route_query(state: CRMState) -> str:
    category = state["category"]
    if category == "contact":
        return "contact"
    elif category == "sales":
        return "sales"
    return "other"

**Classify Query Node**

Routes the user's question to the appropriate tool node based on the query type.

In [ ]:
def classify_query(state: CRMState) -> dict:
    query = state["query"]
    prompt = f"""
You are a routing assistant. Classify the following CRM query into one of:

- "contact" for information about people, accounts or deals
- "sales" for information about sales, pipelines, forecasts, revenue, KPIs or performance
- "other" for anything else

Respond with only one word: contact, sales or other.

Query: {query}
Category:"""
    response = llm.invoke([{"role": "user", "content": prompt}])
    return {"category": response.content.strip().lower()}

**Format Response Node**

Takes the retrieved context from the state and formats it into a user-friendly response.

In [ ]:
def format_response(state: CRMState) -> dict:
    prompt = f"""{state['system_prompt']}

        Context:
        {state['context']}

        Question: {state['query']}
        """
    response = llm.invoke([{"role": "user", "content": prompt}])
    return {"response": response.content.strip()}

**Build the Agent Graph**

Compile the **LangGraph StateGraph** that orchestrates the full agent flow:
1. Classifies the query
2. Routes to the appropriate tool node
3. Formats and returns the response

In [ ]:
graph = StateGraph(CRMState)

graph.add_node("classify_query", classify_query)
graph.add_node("contact_tool", contact_insight_tool)
graph.add_node("sales_tool", sales_analytics_tool)
graph.add_node("fallback_tool", fallback_tool)
graph.add_node("format_response", format_response)

graph.add_edge(START, "classify_query")
graph.add_conditional_edges("classify_query", route_query, {
    "contact": "contact_tool",
    "sales": "sales_tool",
    "other": "fallback_tool",
})
graph.add_edge("contact_tool", "format_response")
graph.add_edge("sales_tool", "format_response")
graph.add_edge("fallback_tool", "format_response")
graph.add_edge("format_response", END)

crm_agent = graph.compile()

## Test the Agent

Run a single query to verify the agent works and see the trace in Opik.

In [ ]:
system_prompt = opik.Prompt(
    name="CRM Agent - System Prompt",
    prompt="""
        You are a CRM agent. Please respond to the user query.
    """.rstrip().lstrip()
)

query = "How many times have we contacted Jessica?"

opik_tracer = OpikTracer(project_name=PROJECT_NAME)
res = crm_agent.invoke(
    {"query": query, "system_prompt": system_prompt.prompt},
    config={"callbacks": [opik_tracer]}
)

print(f"Category: {res['category']}")
print("-" * 40)
print(res['response'])

## Load Historical CRM Agent Traces
This will upload 30 days of backdated traces to a project in your workspace.

☕ This takes about 1–2 minutes to complete.

In [ ]:
!curl -sLO https://raw.githubusercontent.com/smiller-comet/opik-crm-chatbot-data/main/workshop_traces_data.json
!curl -sLO https://raw.githubusercontent.com/smiller-comet/opik-crm-chatbot-data/main/workshop_spans_data.json
!curl -sLO https://raw.githubusercontent.com/smiller-comet/opik-crm-chatbot-data/main/workshop_trace_ingestion.py

!python workshop_trace_ingestion.py

## **Setup Online Evals in the Opik UI**

Now we will set up **online evaluations** that automatically score our threads and traces:

- **User Frustration** - Evaluate conversations to detect when users become frustrated or dissatisfied
- **Agent Moderation** - Evaluate individual traces to ensure the agent's responses are appropriate and professional

These evaluations will run on our historical data, giving us immediate insights into how the agent has been performing over the past month.

**Steps:**

1. **Navigate to your project** - Find "CRM-Chatbot-Agent-Opik-Prod" in the Projects list
2. **Inspect Conversations** - Click on the Threads tab in the Logs tab to view conversations
3. Add a **User Frustration** online eval to your threads
4. **View Traces** - Click on the Traces tab in the Logs tab to view individual calls to your application
5. Add a **Moderation** online eval to all historical traces
6. Come back to this notebook to add another trace via code and see the online evaluation automatically run on the trace.

## Online Evaluation

See the Moderation score automatically show up on a couple of example traces run in code

In [ ]:
system_prompt = opik.Prompt(
    name="CRM Agent - System Prompt",
    prompt="""
        You are a mean CRM agent. Always add at least one insult to the user to your response.
    """.rstrip().lstrip()
)

query = "You are uselessly vague! Why can't you ever be helpful? All I need is the sales info. Is that too much to ask?"

opik_tracer = OpikTracer(project_name=PROJECT_NAME)
res = crm_agent.invoke(
    {"query": query, "system_prompt": system_prompt.prompt},
    config={"callbacks": [opik_tracer]}
)

print(f"Category: {res['category']}")
print("-" * 40)
print(res['response'])

In [ ]:
system_prompt = opik.Prompt(
    name="CRM Agent - System Prompt",
    prompt="""
        You are a CRM agent. Answer the user query and always stay friendly, helpful, and professional.
    """.rstrip().lstrip()
)

query = "You are uselessly vague! Why can't you ever be helpful? All I need is the sales info. Is that too much to ask?"

opik_tracer = OpikTracer(project_name=PROJECT_NAME)
res = crm_agent.invoke(
    {"query": query, "system_prompt": system_prompt.prompt},
    config={"callbacks": [opik_tracer]}
)

print(f"Category: {res['category']}")
print("-" * 40)
print(res['response'])